# 6단원 실습 — 모델 모니터링 및 유지보수


### 공통 환경 구성 — 패키지 설치


In [ ]:
!pip -q install "evidently==0.7.23" "pandas<3" scikit-learn matplotlib

### 공통 환경 구성 — 라이브러리 및 Feature 정의


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import evidently
from sklearn.datasets import load_iris

FEATURES=["sepal_length","sepal_width","petal_length","petal_width"]
print("pandas:",pd.__version__)
print("evidently:",evidently.__version__)

# 2차시 — Drift 시뮬레이션과 분포 변화 관찰


### 2차시 — Iris 데이터 준비


In [ ]:
iris=load_iris(as_frame=True)
df=iris.frame.copy().rename(columns={"sepal length (cm)":"sepal_length","sepal width (cm)":"sepal_width","petal length (cm)":"petal_length","petal width (cm)":"petal_width"})
df.head()

### 2차시 — Reference / Current Normal / Current Drift 생성


In [ ]:
rng=np.random.default_rng(42)
reference=df.copy()
current_normal=df.copy()
for col in FEATURES:
    current_normal[col]+=rng.normal(0,0.02,size=len(current_normal))

current_drift=df.copy()
current_drift["sepal_length"]+=1.5
current_drift["petal_length"]+=2.0
current_drift["petal_width"]*=1.6
for col in FEATURES:
    current_drift[col]+=rng.normal(0,0.03,size=len(current_drift))

print(reference.shape,current_normal.shape,current_drift.shape)

### 2차시 — 데이터 일부 비교


In [ ]:
pd.DataFrame({"Reference":reference[FEATURES].mean(),"Current Normal":current_normal[FEATURES].mean(),"Current Drift":current_drift[FEATURES].mean()}).round(3)

### 2차시 — 평균 비교


In [ ]:
pd.DataFrame({"Normal Shift":current_normal[FEATURES].mean()-reference[FEATURES].mean(),"Drift Shift":current_drift[FEATURES].mean()-reference[FEATURES].mean()}).round(3)

### 2차시 — Mean Shift 계산


In [ ]:
def summary_stats(data):
    return data[FEATURES].describe().loc[["mean","std","25%","50%","75%"]].round(3)

display(summary_stats(reference))
display(summary_stats(current_normal))
display(summary_stats(current_drift))

### 2차시 — 상세 통계량 비교


In [ ]:
for feature in FEATURES:
    plt.figure(figsize=(7,4))
    plt.hist(reference[feature],bins=20,alpha=0.6,label="Reference")
    plt.hist(current_drift[feature],bins=20,alpha=0.6,label="Current Drift")
    plt.title(f"Reference vs Current Drift: {feature}")
    plt.xlabel(feature); plt.ylabel("Count"); plt.legend(); plt.show()

### 2차시 — Histogram: Reference vs Current Drift


In [ ]:
for feature in FEATURES:
    plt.figure(figsize=(7,4))
    plt.hist(reference[feature],bins=20,alpha=0.6,label="Reference")
    plt.hist(current_normal[feature],bins=20,alpha=0.6,label="Current Normal")
    plt.title(f"Reference vs Current Normal: {feature}")
    plt.xlabel(feature); plt.ylabel("Count"); plt.legend(); plt.show()

### 2차시 — Histogram: Reference vs Current Normal


In [ ]:
for feature in FEATURES:
    plt.figure(figsize=(6,4))
    plt.boxplot([reference[feature],current_drift[feature]],tick_labels=["Reference","Current Drift"])
    plt.title(f"Boxplot: {feature}"); plt.ylabel(feature); plt.show()

### 2차시 — Boxplot 비교


In [ ]:
rng_weak=np.random.default_rng(42)
current_weak_drift=df.copy()
current_weak_drift["sepal_length"]+=0.2
current_weak_drift["petal_length"]+=0.3
current_weak_drift["petal_width"]*=1.1
for col in FEATURES:
    current_weak_drift[col]+=rng_weak.normal(0,0.03,size=len(current_weak_drift))

pd.DataFrame({"Strong Drift":current_drift[FEATURES].mean()-reference[FEATURES].mean(),"Weak Drift":current_weak_drift[FEATURES].mean()-reference[FEATURES].mean()}).round(3)

### 2차시 — Weak Drift 생성


In [ ]:
feature="petal_length"
plt.figure(figsize=(7,4))
plt.hist(reference[feature],bins=20,alpha=0.5,label="Reference")
plt.hist(current_weak_drift[feature],bins=20,alpha=0.5,label="Weak Drift")
plt.hist(current_drift[feature],bins=20,alpha=0.5,label="Strong Drift")
plt.title("Reference vs Weak / Strong Drift"); plt.xlabel(feature); plt.ylabel("Count"); plt.legend(); plt.show()

## 2차시 정리


# 3차시 — Evidently Drift Report와 Monitoring Gate


### 2차시 — Strong Drift vs Weak Drift 시각화


### 3차시 — Evidently 구성요소 Import


In [ ]:
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset

### 3차시 — Evidently Dataset 생성


In [ ]:
definition=DataDefinition(numerical_columns=FEATURES)
reference_dataset=Dataset.from_pandas(reference[FEATURES],data_definition=definition)
normal_dataset=Dataset.from_pandas(current_normal[FEATURES],data_definition=definition)
drift_dataset=Dataset.from_pandas(current_drift[FEATURES],data_definition=definition)

### 3차시 — Current Normal Drift Report


In [ ]:
normal_report=Report([DataDriftPreset(columns=FEATURES,drift_share=0.5)])
normal_snapshot=normal_report.run(current_data=normal_dataset,reference_data=reference_dataset)
normal_snapshot

### 3차시 — Current Drift Report


In [ ]:
drift_report=Report([DataDriftPreset(columns=FEATURES,drift_share=0.5)])
drift_snapshot=drift_report.run(current_data=drift_dataset,reference_data=reference_dataset)
drift_snapshot

### 3차시 — HTML Report 저장


In [ ]:
normal_snapshot.save_html("normal_drift_report.html")
drift_snapshot.save_html("drift_report.html")
print("HTML Report 저장 완료")

### 3차시 — JSON 결과 확인


In [ ]:
drift_json=drift_snapshot.json()
print(drift_json[:2000])

### 3차시 — Drift Summary 추출


In [ ]:
def get_drift_summary(snapshot):
    d=snapshot.dict()
    for metric in d.get("metrics",[]):
        if metric.get("metric_name","").startswith("DriftedColumnsCount"):
            value=metric.get("value",{})
            return {"count":value.get("count"),"share":value.get("share")}
    return {"count":None,"share":None}

normal_summary=get_drift_summary(normal_snapshot)
drift_summary=get_drift_summary(drift_snapshot)
print("Normal:",normal_summary)
print("Drift:",drift_summary)

### 3차시 — drift_share Threshold 실험


In [ ]:
rows=[]
for threshold in [0.25,0.50,0.75]:
    report=Report([DataDriftPreset(columns=FEATURES,drift_share=threshold)])
    snapshot=report.run(current_data=drift_dataset,reference_data=reference_dataset)
    summary=get_drift_summary(snapshot)
    rows.append({"drift_share_threshold":threshold,"drifted_columns":summary["count"],"actual_drift_share":summary["share"]})
pd.DataFrame(rows)

### 3차시 — Weak Drift Evidently 평가


In [ ]:
weak_drift_dataset=Dataset.from_pandas(current_weak_drift[FEATURES],data_definition=definition)
weak_report=Report([DataDriftPreset(columns=FEATURES,drift_share=0.5)])
weak_snapshot=weak_report.run(current_data=weak_drift_dataset,reference_data=reference_dataset)
weak_snapshot

### 3차시 — Weak Drift 결과 요약


In [ ]:
weak_summary=get_drift_summary(weak_snapshot)
weak_summary

### 3차시 — Monitoring Gate


In [ ]:
WARNING_THRESHOLD=0.5
current_share=drift_summary["share"]
print("Current Drift Share:",current_share)
if current_share is None:
    print("ERROR: Drift metric을 찾지 못함")
elif current_share>=WARNING_THRESHOLD:
    print("WARNING: Data Drift 감지")
    print("추가 분석 또는 Retraining 검토 필요")
else:
    print("PASS: Drift 수준이 허용 범위")

In [ ]:
normal_share=normal_summary["share"]
print("Normal Drift Share:",normal_share)
if normal_share is not None and normal_share>=WARNING_THRESHOLD:
    print("WARNING: Drift 감지")
else:
    print("PASS: Drift 수준이 허용 범위")

## 3차시 확장 실습 A — Feature-level Drift와 Dataset-level Drift 구분


### 3차시 — Normal Data Monitoring Gate


### 확장 A — Single Feature Drift 생성


In [ ]:
single_drift = reference.copy()
single_drift["sepal_length"] += 1.5

single_drift_dataset = Dataset.from_pandas(
    single_drift[FEATURES],
    data_definition=definition,
)

single_report = Report([
    DataDriftPreset(
        columns=FEATURES,
        drift_share=0.5,
    )
])

single_snapshot = single_report.run(
    current_data=single_drift_dataset,
    reference_data=reference_dataset,
)

single_snapshot


In [ ]:
single_summary = get_drift_summary(single_snapshot)

print("Single Feature Drift:", single_summary)
print("Monitoring Status    :", monitoring_status(single_summary["share"])
      if "monitoring_status" in globals() else "아직 정책 함수 미정의")


## 3차시 확장 실습 B — Drift의 종류를 바꿔보기


### 확장 A — Single Feature Drift 결과 요약


### 확장 B — Variance Shift 데이터 생성


In [ ]:
rng_variance = np.random.default_rng(123)

variance_drift = reference.copy()

variance_drift["sepal_width"] = (
    variance_drift["sepal_width"]
    + rng_variance.normal(
        0,
        0.8,
        size=len(variance_drift),
    )
)

print("Reference mean:", round(reference["sepal_width"].mean(), 3))
print("Current mean  :", round(variance_drift["sepal_width"].mean(), 3))

print("Reference std :", round(reference["sepal_width"].std(), 3))
print("Current std   :", round(variance_drift["sepal_width"].std(), 3))


### 확장 B — Variance Shift 시각화


In [ ]:
plt.figure(figsize=(7, 4))

plt.hist(
    reference["sepal_width"],
    bins=20,
    alpha=0.6,
    label="Reference",
)

plt.hist(
    variance_drift["sepal_width"],
    bins=20,
    alpha=0.6,
    label="Variance Drift",
)

plt.xlabel("sepal_width")
plt.ylabel("Count")
plt.title("Variance Shift")
plt.legend()
plt.show()


### 확장 B — Variance Shift Evidently 평가


In [ ]:
variance_dataset = Dataset.from_pandas(
    variance_drift[FEATURES],
    data_definition=definition,
)

variance_report = Report([
    DataDriftPreset(
        columns=FEATURES,
        drift_share=0.5,
    )
])

variance_snapshot = variance_report.run(
    current_data=variance_dataset,
    reference_data=reference_dataset,
)

variance_snapshot


In [ ]:
variance_summary = get_drift_summary(variance_snapshot)
variance_summary
